# Reencuadre fenológico — extensión completa del baseline

Este cuaderno extiende el baseline tabular del `04_baseline.ipynb` con todo lo que el proyecto tiene programado y testeado: bloques opcionales del fused, modelos temporales reales, clustering sobre la firma fenológica pura, estrategia para el desbalance, rama semántica LLM y glosario para el entregable.

**Preguntas oficiales del Avance 3 a las que aporta**:

- **P1 (algoritmo baseline)**: compara los 3 modelos tabulares (RF + XGB + LightGBM) contra los 2 modelos temporales reales (TempCNN + InceptionTime) sobre el mismo conjunto ganador.
- **P2 (importancia / features irrelevantes)**: amplía la ablación de `04c_baseline.ipynb` con los 3 bloques opcionales (FarSLIP, pheno_text, REP) y decide promover/descartar cada uno.
- **P3 (sub/sobreajuste)**: los modelos temporales se diagnostican leyendo la curva train_loss vs val_loss desde MLflow con `diagnose_temporal_fit`.

Estructura del cuaderno (10 secciones):

1. Carga del dataset base y materialización de los bloques opcionales (sin skips silenciosos).
2. Fusión: base + FarSLIP + pheno_text + spectral_signature.
3. Ablación con todos los bloques opcionales (lee el conjunto ganador BASE producido por `04c_baseline.ipynb`).
4. Comparativa de los 5 modelos (RF + XGB + LGBM + TempCNN + InceptionTime) sobre el conjunto ganador.
5. Diagnóstico por clase del mejor modelo temporal: matriz de confusión OOF + F1 por clase.
6. Clustering KMeans sobre la firma fenológica pura + UMAP 2D + curvas NDVI medias por cluster (sin coordenadas).
7. Estrategia para el desbalance ~31× max/min.
8. Rama semántica fenológica: descripción textual real con Gemini Flash sobre subset balanceado.
9. Conclusiones consolidadas con decisión por bloque.
10. Glosario para el entregable del curso.

In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FUSED_PATH = "data/features/features_fused_italy.parquet"
PHENO_TEXT_PATH = "data/features/phenology_text_italy.parquet"
# Output paths renombrados a _pastis_2019 en US-023-preview v2: los
# parquets viejos (_italy) eran cache vacio (NaN al 100%) por el
# bug B04 vs B4 en GEE y fallback DOY estatico. Los nuevos usan
# bandas correctas + anclas calendario por parcela.
S2_ANCHORS_PATH = "data/features/s2_anchors_pastis_2019.parquet"
SPECTRAL_SIGNATURE_PATH = "data/features/spectral_signature_pastis_2019.parquet"
FARSLIP_PATH = "data/farslip/embeddings_italy.parquet"
ALPHAEARTH_ENRICHED_PATH = "data/cache/gee/alphaearth_pastis_parcels_2019_85951_enriched.parquet"
PASTIS_METADATA_GEOJSON = "data/PASTIS-R/metadata.geojson"
PHENOLOGY_ANCHORS_PATH = "data/features/pastis_phenology_anchors_2019.parquet"
BASE_ABLATION_PATH = "reports/baseline/04c_baseline/ablation_table.parquet"
FIGURES_SUBDIR = "us-023-preview/05_reencuadre"
REPORTS_SUBDIR = "baseline/05_reencuadre"
YEAR = 2023
K_FOLDS = 5
BUFFER_KM = 1.0
MAX_SAMPLES = None
RANDOM_STATE = 42
# FarSLIP embeddings_italy.parquet contiene parcelas italianas extra-PASTIS
# (overlap parcel_id con PASTIS = 0). Integrar FarSLIP a PASTIS requiere
# distillar el student (US-017) y adaptar vocabulary CAP a cultivos
# franceses. Por ahora se excluye del fused PASTIS; su evaluacion honesta
# vive en 04_farslip_eval_pastis.ipynb sobre el subset adecuado.
ENABLE_FARSLIP = False
ENABLE_PHENO_TEXT = True
ENABLE_SPECTRAL_SIGNATURE = True
ENABLE_ALPHAEARTH = True  # Anexa data/cache/gee/alphaearth_*_enriched.parquet (64 dim_NN)
ENABLE_TEMPORAL_MODELS = True
# Si True, reusa los runs MLflow ya finalizados de tempcnn/inceptiontime
# en el experimento baseline-05-reencuadre y evita re-entrenarlos (~6h GPU).
REUSE_TEMPORAL_FROM_MLFLOW = True
ENABLE_CLUSTERING = True
ENABLE_LLM_BRANCH = True
ENFORCE_GEMINI_API_KEY = True
TEMPORAL_EPOCHS = 80
TEMPORAL_BATCH_SIZE = 256
TEMPORAL_DEVICE = 'auto'  # auto = cuda si disponible
N_CLUSTERS = 8
LLM_SUBSET_SIZE = 1080  # 60 parcelas balanceadas x 18 clases
WEAK_CLASS_THRESHOLD = 1000  # parcelas, para listar clases debiles
MLFLOW_EXPERIMENT = "baseline-05-reencuadre"


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))

# Chdir al repo root para que las rutas relativas de la celda `parameters`
# (FEATURES_PATH = "data/...", etc.) resuelvan igual sin importar desde donde
# se haya lanzado el kernel (VS Code abre con cwd = carpeta del notebook).
# Esto preserva el contrato papermill (parametros como strings relativas) y
# elimina FileNotFoundError causado por cwd != repo root.
os.chdir(env.repo)
display(Markdown(f"**cwd anclado al repo root**: `{env.repo}`"))


### Trazabilidad MLflow

Cada bloque que entrena un modelo abre runs MLflow propios agrupados en el experimento `baseline-05-reencuadre`: ablación, 5 modelos sobre conjunto ganador (RF, XGB, LGBM, TempCNN, InceptionTime) y rama LLM. Los `run_id` se acumulan en el dict `MLFLOW_RUN_IDS` para que `Avance3.Equipo17.ipynb` consolide la trazabilidad.

In [ ]:
from ml.utils.mlflow_utils import (
    resolve_tracking_uri,
    track_experiment,
    server_is_reachable,
)

# Resolucion robusta del tracking URI: si MLFLOW_TRACKING_URI esta en
# `.env.local` pero el server no responde (Docker apagado, contenedor
# detenido), caemos a `file:./mlruns` para no detener el notebook.
_candidate_uri = resolve_tracking_uri(None, probe_server=False)
if _candidate_uri.startswith(('http://', 'https://')) and not server_is_reachable(_candidate_uri):
    mlflow_uri = 'file:./mlruns'
    display(Markdown(
        f'> Servidor MLflow `{_candidate_uri}` no responde. '
        'Caigo a tracking local `file:./mlruns`. '
        'Para registrar en el server, ejecuta `docker compose up -d mlflow` '
        'antes de re-ejecutar las celdas MLflow.'
    ))
else:
    mlflow_uri = _candidate_uri
display(Markdown(
    f'**MLflow tracking URI**: `{mlflow_uri}` · '
    f'**Experimento**: `{MLFLOW_EXPERIMENT}`'
))
MLFLOW_RUN_IDS: dict[str, str] = {}


## 1. Carga del dataset base (con metadata)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    materialize_phenology_text_if_missing,
    materialize_s2_anchors_if_missing,
    materialize_spectral_signature_if_missing,
    run_ablation_and_persist,
)
from ml.utils.parcel_id import canonical_parcel_id
from ml.eval.reencuadre_plots import (
    plot_ablation_bars,
    plot_optional_blocks_ablation,
)
from ml.eval.feature_ablation import FeatureAblationResult
from pathlib import Path

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
display(Markdown(f'**Dataset base**: `{df.height:,}` parcelas x `{df.width}` cols'))


## Materialización del bloque `pheno_text` (Gemini sobre el dataset completo)

In [ ]:
if ENABLE_PHENO_TEXT:
    if ENFORCE_GEMINI_API_KEY and not env.has_gemini_api_key:
        raise RuntimeError(
            'GEMINI_API_KEY ausente. Define la variable en `.env.local` antes de re-ejecutar, '
            'o pon ENFORCE_GEMINI_API_KEY=False para correr solo las ablaciones base.'
        )
    pheno_path = materialize_phenology_text_if_missing(
        parcels_features_path=FEATURES_PATH,
        output_path=PHENO_TEXT_PATH,
        enforce_api_key=ENFORCE_GEMINI_API_KEY,
    )
    pheno_df = canonical_parcel_id(pl.read_parquet(pheno_path))
    display(Markdown(f'**pheno_text**: `{pheno_df.shape}` en `{pheno_path}`'))
else:
    pheno_df = None
    display(Markdown('> ENABLE_PHENO_TEXT=False: bloque omitido.'))


## Materialización de anclas Sentinel-2 y firma espectral REP (Frampton 2013)

In [ ]:
from ml.ingest.pastis_phenology_anchors import build_pastis_phenology_anchors

if ENABLE_SPECTRAL_SIGNATURE:
    if not env.has_ee_credentials:
        display(Markdown(
            '> Earth Engine no configurado. Define `GEE_PROJECT_ID` '
            'en `.env.local` o ejecuta `earthengine authenticate`. '
            'El muestreo S2 anchors fallara sin esto.'
        ))
    # Anclas fenologicas por parcela en DOY calendario 2019.
    # Si el parquet existe usa cache; si no, lo construye desde
    # `metadata.geojson` PASTIS-R + DOY relativos del subset US-016.
    phen_anchors_path = build_pastis_phenology_anchors(
        metadata_geojson_path=PASTIS_METADATA_GEOJSON,
        features_subset_path=FEATURES_PATH,
        output_path=PHENOLOGY_ANCHORS_PATH,
        target_year=YEAR,
    )
    display(Markdown(
        f'**Anclas fenologicas por parcela**: `{phen_anchors_path}` '
        '(DOY calendario derivado de `metadata.geojson` PASTIS-R).'
    ))
    anchors_path = materialize_s2_anchors_if_missing(
        parcels_geoparquet=PARCELS_GEOPARQUET,
        output_path=S2_ANCHORS_PATH,
        year=YEAR,
        phenology_anchors_path=phen_anchors_path,
    )
    spec_path = materialize_spectral_signature_if_missing(
        s2_anchors_path=anchors_path,
        output_path=SPECTRAL_SIGNATURE_PATH,
        descriptor='rep',
    )
    spec_df = canonical_parcel_id(pl.read_parquet(spec_path))
    display(Markdown(f'**spectral_signature**: `{spec_df.shape}` en `{spec_path}`'))
else:
    spec_df = None
    display(Markdown('> ENABLE_SPECTRAL_SIGNATURE=False: bloque omitido.'))


## Carga de FarSLIP desde la ruta canónica (`parcel_id` en Utf8)

In [ ]:
if ENABLE_FARSLIP:
    farslip_path = Path(FARSLIP_PATH)
    if not farslip_path.exists():
        raise FileNotFoundError(
            f'FarSLIP parquet no encontrado en {farslip_path}. '
            'Ejecuta `dvc pull data/farslip/embeddings_italy.parquet.dvc` '
            'antes de re-ejecutar.'
        )
    farslip_df = canonical_parcel_id(pl.read_parquet(farslip_path))
    display(Markdown(f'**FarSLIP**: `{farslip_df.shape}` en `{farslip_path}` con parcel_id Utf8.'))
else:
    farslip_df = None
    display(Markdown('> ENABLE_FARSLIP=False: bloque omitido.'))


## Carga del bloque AlphaEarth (64 dim_NN) sobre PASTIS

In [ ]:
if ENABLE_ALPHAEARTH:
    ae_path = Path(ALPHAEARTH_ENRICHED_PATH)
    if not ae_path.exists():
        raise FileNotFoundError(
            f'AlphaEarth enriched parquet no encontrado en {ae_path}. '
            'Ejecuta el pipeline GEE (US-012) o `dvc pull` del cache.'
        )
    ae_df = canonical_parcel_id(pl.read_parquet(ae_path))
    n_dims = sum(1 for c in ae_df.columns if c.startswith('dim_'))
    display(Markdown(
        f'**AlphaEarth**: `{ae_df.shape}` en `{ae_path}` '
        f'con `{n_dims}` dimensiones `dim_NN` reales.'
    ))
else:
    ae_df = None
    display(Markdown('> ENABLE_ALPHAEARTH=False: bloque omitido.'))


## Fusión de bloques: base + AlphaEarth + pheno_text + spectral_signature

Aplicamos un LEFT JOIN secuencial sobre `parcel_id` (todos en Utf8 tras `canonical_parcel_id`). AlphaEarth tiene overlap 100% con el subset PASTIS (mismo dataset GEE 2019, 85951 parcelas). Los bloques `pheno_text` y `spectral_signature` cubren un subset menor — las parcelas sin coincidencia quedan con NaN; XGBoost y LightGBM los toleran nativamente y RandomForest los imputa por mediana. **FarSLIP** se excluye del fused PASTIS porque `embeddings_italy.parquet` contiene parcelas italianas extra-PASTIS (overlap=0); su evaluación honesta vive en `04_farslip_eval_pastis.ipynb`.

In [ ]:
df = canonical_parcel_id(df)
fused = df
joined_log = []
if ae_df is not None:
    keep = ['parcel_id'] + [c for c in ae_df.columns if c.startswith('dim_')]
    fused = fused.join(ae_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'AlphaEarth: +{len(keep)-1} cols')
if farslip_df is not None:
    keep = ['parcel_id'] + [c for c in farslip_df.columns if c.startswith('farslip_')]
    fused = fused.join(farslip_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'FarSLIP: +{len(keep)-1} cols')
if pheno_df is not None:
    keep = ['parcel_id'] + [c for c in pheno_df.columns if c.startswith('pheno_text_')]
    fused = fused.join(pheno_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'pheno_text: +{len(keep)-1} cols')
if spec_df is not None:
    keep = ['parcel_id'] + [c for c in spec_df.columns if c.startswith('spectral_signature_')]
    fused = fused.join(spec_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'spectral_signature: +{len(keep)-1} cols')

display(Markdown(
    f"**Conjunto fused final**: `{fused.shape}`\n\n"
    + "\n".join(f"- {l}" for l in joined_log)
))


## 3. Ablación con todos los bloques opcionales (sobre el conjunto fused)

In [ ]:
ablation_table, parquet_path = run_ablation_and_persist(
    fused,
    output_dir=env.reports_dir,
    models=('xgb',),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    max_samples=MAX_SAMPLES,
)
display(Markdown(f'**Tabla de ablación**: `{parquet_path.relative_to(env.repo)}`'))
display(ablation_table)

# MLflow: un run por feature_set evaluado en la ablacion opcional.
import math
for row in ablation_table.iter_rows(named=True):
    fs = row['feature_set']
    with track_experiment(
        experiment_name=MLFLOW_EXPERIMENT,
        run_name=f'05-ablation-{fs}',
        tracking_uri=mlflow_uri,
        dvc_path=FEATURES_PATH,
        probe_server=False,
    ) as run:
        import mlflow
        mlflow.log_params({
            'feature_set': fs,
            'model_kind': row['model'],
            'n_features': row['n_features'],
            'k_folds': K_FOLDS,
            'buffer_km': BUFFER_KM,
            'has_farslip': ENABLE_FARSLIP,
            'has_pheno_text': ENABLE_PHENO_TEXT,
            'has_spectral_signature': ENABLE_SPECTRAL_SIGNATURE,
        })
        for key in ('f1_macro', 'f1_weighted', 'miou', 'delta_vs_full'):
            val = row.get(key)
            if val is not None and not (isinstance(val, float) and math.isnan(val)):
                mlflow.log_metric(key, float(val))
        MLFLOW_RUN_IDS[f'ablation-{fs}'] = run.info.run_id


### 3.1 Gráficos: ablación completa, fuga geométrica y aporte de bloques opcionales

In [ ]:
results = [
    FeatureAblationResult(
        feature_set=row['feature_set'],
        model_kind=row['model'],
        f1_macro=row['f1_macro'] if row['f1_macro'] is not None else float('nan'),
        f1_weighted=row['f1_weighted'] if row['f1_weighted'] is not None else float('nan'),
        miou=row['miou'] if row['miou'] is not None else float('nan'),
        n_features=row['n_features'],
        delta_vs_full=row['delta_vs_full'] if row['delta_vs_full'] is not None else float('nan'),
    )
    for row in ablation_table.iter_rows(named=True)
]

fig_abl = plot_ablation_bars(results, title='F1-macro por conjunto (ablación completa)')
fig_abl.savefig(env.figures_dir / 'ablation_full.png', bbox_inches='tight')
display(fig_abl)
plt.close(fig_abl)

# Nota: el plot `geom_leakage` se genera SOLO en 04c (donde vive la
# ablacion base con geom_only). Aqui mostramos solo el aporte de los
# bloques opcionales para no duplicar contenido entre notebooks.
fig_opt = plot_optional_blocks_ablation(results)
fig_opt.savefig(env.figures_dir / 'optional_blocks.png', bbox_inches='tight')
display(fig_opt)
plt.close(fig_opt)


## 4. Comparativa de los 5 modelos sobre el conjunto ganador

Tres modelos tabulares (Random Forest, XGBoost, LightGBM) y dos modelos temporales (TempCNN, InceptionTime) entrenados sobre el **mismo conjunto ganador post-ablación**, con la misma validación cruzada espacial. La elección del ganador se delega a `04c_baseline.ipynb`: si su `ablation_table.parquet` está disponible se usa; si no, fallback a `no_geom` (decisión documentada).

- **Tabulares** (RF/XGB/LGBM): ven el vector resumen espectro-temporal anual (features estadísticas + FFT).
- **Temporales** (TempCNN/InceptionTime): leen la curva NDVI/NDWI/EVI completa reconstruida a T=72 muestras (importados de `breizhcrops.models`, no reimplementados).


In [ ]:
from ml.train.baseline import train_one_model
from ml.train.phenology_models import train_temporal_model

# Lee el conjunto ganador BASE de 04c. Si no existe, fallback no_geom.
base_ablation_path = env.repo / BASE_ABLATION_PATH
if base_ablation_path.exists():
    base_table = pl.read_parquet(base_ablation_path)
    finite = base_table.filter(pl.col('f1_macro').is_finite()).sort('f1_macro', descending=True)
    winner_base = finite.row(0, named=True)['feature_set'] if finite.height else 'no_geom'
    display(Markdown(f'**Conjunto ganador base** (desde 04c): `{winner_base}`'))
else:
    winner_base = 'no_geom'
    display(Markdown(
        f'> Tabla `{BASE_ABLATION_PATH}` no encontrada. '
        'Fallback documentado: `no_geom` (descarta las 3 columnas `geom_*` por leakage).'
    ))

# Aplicar el filtro al fused (descartar geom_* si winner_base == 'no_geom').
def _apply_winner(df_in, winner_set):
    if winner_set == 'no_geom':
        return df_in.drop([c for c in df_in.columns if c.startswith('geom_')])
    return df_in
fused_winner = _apply_winner(fused, winner_base)
display(Markdown(
    f'**Dataset para los 5 modelos**: `{fused_winner.shape}` '
    f'(winner_base=`{winner_base}`).'
))


### 4.1 Entrenamiento de los 3 modelos tabulares con MLflow

Cada modelo tabular abre un run MLflow propio. Tiempo esperado: 30-60 min en total sobre RTX 4070 (XGB y LGBM en CPU paralelo, RF en CPU multinúcleo).

In [ ]:
tabular_results = {}
for kind in ('rf', 'xgb', 'lgbm'):
    with track_experiment(
        experiment_name=MLFLOW_EXPERIMENT,
        run_name=f'05-winner-{kind}',
        tracking_uri=mlflow_uri,
        dvc_path=FEATURES_PATH,
        probe_server=False,
    ) as run:
        res = train_one_model(
            fused_winner, model=kind,
            k_folds=K_FOLDS, buffer_km=BUFFER_KM,
            random_state=RANDOM_STATE,
        )
        tabular_results[kind] = res
        import mlflow
        mlflow.log_params({
            'model': kind,
            'winner_base': winner_base,
            'k_folds': K_FOLDS,
            'n_parcels': fused_winner.height,
            'n_features': len(res.feature_cols),
        })
        mlflow.log_metrics({
            'f1_macro': res.metrics['f1_macro'],
            'f1_weighted': res.metrics['f1_weighted'],
            'miou': res.metrics['miou'],
            'accuracy': res.metrics['accuracy'],
            'kappa': res.metrics['cohen_kappa'],
        })
        MLFLOW_RUN_IDS[f'winner-{kind}'] = run.info.run_id
        display(Markdown(
            f'**`{kind}`** ajustado: F1-macro = `{res.metrics["f1_macro"]:.4f}` '
            f'(run `{run.info.run_id[:12]}...`).'
        ))


### 4.2 Entrenamiento de los 2 modelos temporales (TempCNN + InceptionTime)

Reentrenamiento real sobre la curva NDVI/NDWI/EVI reconstruida (T=72 a partir de la FFT). MLflow loggea `fold{i}_train_loss` y `fold{i}_val_loss` por época — recuperaremos esas curvas para diagnosticar sub/sobreajuste en la sección 5. Tiempo esperado: 45-90 min en GPU RTX 4070.

In [ ]:
from ml.utils.baseline_notebook_helpers import load_temporal_result_from_mlflow

temporal_results = {}
if ENABLE_TEMPORAL_MODELS:
    for kind in ('tempcnn', 'inceptiontime'):
        res = None
        if REUSE_TEMPORAL_FROM_MLFLOW:
            try:
                res = load_temporal_result_from_mlflow(
                    kind,
                    experiment_name=MLFLOW_EXPERIMENT,
                    tracking_uri=mlflow_uri,
                )
                display(Markdown(
                    f'**`{kind}`** recuperado de MLflow: F1-macro = `{res.f1_macro:.4f}` '
                    f'(run `{res.mlflow_run_id[:12]}...`, sin re-entrenar).'
                ))
            except ValueError as exc:
                display(Markdown(
                    f'> Cache MLflow no disponible para `{kind}`: {exc}. Entreno desde cero.'
                ))
        if res is None:
            # train_temporal_model abre su propio run MLflow.
            res = train_temporal_model(
                df=fused_winner,
                model_kind=kind,
                n_epochs=TEMPORAL_EPOCHS,
                batch_size=TEMPORAL_BATCH_SIZE,
                device=TEMPORAL_DEVICE,
                mlflow_uri=mlflow_uri,
                k_folds=K_FOLDS,
                buffer_km=BUFFER_KM,
                seed=RANDOM_STATE,
            )
            display(Markdown(
                f'**`{kind}`** entrenado: F1-macro = `{res.f1_macro:.4f}` '
                f'(run `{(res.mlflow_run_id or "local")[:12]}...`).'
            ))
        temporal_results[kind] = res
        if res.mlflow_run_id:
            MLFLOW_RUN_IDS[f'winner-{kind}'] = res.mlflow_run_id
else:
    display(Markdown('> ENABLE_TEMPORAL_MODELS=False: modelos temporales omitidos.'))


### 4.3 Tabla y barplot consolidado de los 5 modelos

In [ ]:
rows_5 = []
for k, r in tabular_results.items():
    rows_5.append({
        'model': k, 'family': 'tabular',
        'f1_macro': r.metrics['f1_macro'],
        'f1_weighted': r.metrics['f1_weighted'],
        'miou': r.metrics['miou'],
        'accuracy': r.metrics['accuracy'],
        'kappa': r.metrics['cohen_kappa'],
    })
for k, r in temporal_results.items():
    rows_5.append({
        'model': k, 'family': 'temporal',
        'f1_macro': r.f1_macro,
        'f1_weighted': r.f1_weighted,
        'miou': r.miou,
        'accuracy': float('nan'),
        'kappa': r.cohen_kappa,
    })
comparison_5 = pl.DataFrame(rows_5).sort('f1_macro', descending=True)
comparison_5_path = env.reports_dir / 'model_comparison_temporal.parquet'
comparison_5.write_parquet(comparison_5_path)
display(Markdown(f'**Tabla guardada**: `{comparison_5_path.relative_to(env.repo)}`'))
display(comparison_5)

# Barplot con palette distinta para tabular vs temporal.
fig, ax = plt.subplots(figsize=(9, 5), dpi=110)
palette = {'tabular': '#4C72B0', 'temporal': '#DD8452'}
for i, row in enumerate(comparison_5.iter_rows(named=True)):
    ax.bar(i, row['f1_macro'], color=palette[row['family']],
           label=row['family'] if i == 0 or row['family'] != comparison_5.row(i-1, named=True)['family'] else None)
ax.set_xticks(range(comparison_5.height))
ax.set_xticklabels(comparison_5['model'].to_list(), rotation=15)
ax.set_ylabel('F1-macro out-of-fold')
ax.set_title(f'5 modelos sobre conjunto ganador `{winner_base}`')
ax.axhline(0.60, color='#888', linestyle='--', linewidth=1, label='target P5 = 0.60')
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='best')
fig.tight_layout()
fig.savefig(env.figures_dir / 'model_comparison_5.png', bbox_inches='tight')
display(fig)
plt.close(fig)


## 5. Diagnóstico por clase del mejor modelo temporal

Tomamos el mejor de los dos temporales y examinamos:

- **F1 por clase**: cuánto acierta el modelo en cada cultivo, con un umbral para marcar las clases débiles (F1 < 0.10).
- **Matriz de confusión OOF**: qué clases se confunden entre sí. El sufijo *out-of-fold* significa que cada predicción viene del fold donde esa parcela quedó en validación — son predicciones honestas sobre el dataset completo.
- **Curva de loss train vs val por época** desde MLflow + `diagnose_temporal_fit` (veredicto explícito sub/sobreajuste).

Las clases débiles son candidatas a fusionarse en una macro-clase `other_minor` en la fase de modelo final.

In [ ]:
from ml.eval.reencuadre_plots import plot_per_class_f1
from ml.eval.metrics import confusion_matrix_figure
from ml.eval.learning_curves import (
    fetch_loss_history_from_mlflow,
    plot_loss_history_from_mlflow,
    diagnose_temporal_fit,
)
from ml.ingest.pastis_loader import PASTIS_R_CLASSES

if temporal_results:
    best_temporal = max(temporal_results.values(), key=lambda r: r.f1_macro)
    display(Markdown(
        f'**Mejor modelo temporal**: `{best_temporal.model_kind}` con '
        f'F1-macro = `{best_temporal.f1_macro:.4f}` sobre '
        f'`{best_temporal.n_parcels:,}` parcelas y `{best_temporal.n_classes}` clases.'
    ))
    if best_temporal.y_true_oof.size > 0:
        # Persistimos OOF.
        import numpy as np
        oof_path = env.reports_dir / f'oof_predictions_{best_temporal.model_kind}.npz'
        np.savez_compressed(
            oof_path,
            y_true=best_temporal.y_true_oof,
            y_pred=best_temporal.y_pred_oof,
        )
        display(Markdown(
            f'**OOF guardado**: `{oof_path.relative_to(env.repo)}` '
            f'({best_temporal.y_true_oof.size:,} predicciones).'
        ))
        # F1 por clase.
        class_names = {
            int(c): PASTIS_R_CLASSES.get(int(c), f'c{int(c)}')
            for c in np.unique(best_temporal.y_true_oof)
        }
        fig_f1 = plot_per_class_f1(
            best_temporal.y_true_oof,
            best_temporal.y_pred_oof,
            class_labels=sorted(class_names.keys()),
            class_names=class_names,
            weak_threshold=0.10,
            title=f'F1 por clase ({best_temporal.model_kind}) out-of-fold',
        )
        fig_f1.savefig(env.figures_dir / f'per_class_f1_{best_temporal.model_kind}.png', bbox_inches='tight')
        display(fig_f1)
        plt.close(fig_f1)
        # Matriz de confusion.
        fig_cm = confusion_matrix_figure(
            best_temporal.y_true_oof,
            best_temporal.y_pred_oof,
            class_labels=sorted(class_names.keys()),
            class_names=class_names,
            normalize='true',
            title=f'Matriz de confusión OOF ({best_temporal.model_kind})',
        )
        fig_cm.savefig(env.figures_dir / f'confusion_matrix_{best_temporal.model_kind}.png', bbox_inches='tight')
        display(fig_cm)
        plt.close(fig_cm)
else:
    display(Markdown('> Sección 5 omitida: no se entrenaron modelos temporales.'))


### 5.1 Curva de loss train vs val por época (los 2 temporales)

In [ ]:
for kind, res in temporal_results.items():
    if not res.mlflow_run_id:
        display(Markdown(f'> `{kind}`: sin run_id MLflow — historial no recuperable.'))
        continue
    try:
        history = fetch_loss_history_from_mlflow(
            res.mlflow_run_id,
            model_kind=kind,
            tracking_uri=mlflow_uri,
        )
        fig_loss = plot_loss_history_from_mlflow(history)
        fig_loss.savefig(env.figures_dir / f'loss_history_{kind}.png', bbox_inches='tight')
        display(fig_loss)
        plt.close(fig_loss)
        try:
            diag = diagnose_temporal_fit(history)
            display(Markdown(
                f'**Diagnóstico `{kind}`**: `{diag.verdict}` '
                f'(gap val−train loss = `{diag.gap:.3f}`).\n\n'
                f'{diag.explanation}'
            ))
        except ValueError as e:
            display(Markdown(f'> Diagnóstico `{kind}` no disponible: {e}'))
    except RuntimeError as e:
        display(Markdown(f'> No se pudo recuperar historial de `{kind}` desde MLflow: {e}'))


## 6. Clustering sin coordenadas — ¿hay estructura en la firma fenológica pura?

Clusterizamos las parcelas usando **solo** la firma fenológica (8 features agronómicos: pico, senescencia, área bajo curva NDVI, etc., más los 24 armónicos FFT de NDVI/NDWI/EVI). **No entran coordenadas, ni `geom_*`, ni clima, ni embedding satelital.**

Si los clusters corresponden a **arquetipos estacionales reconocibles** (cultivo de invierno, cultivo de verano largo, suelo desnudo), la fenología pura organiza el dataset sin necesidad del contexto geográfico. El test visual es UMAP 2D coloreado por cluster + curva NDVI media reconstruida por cluster.

In [ ]:
if ENABLE_CLUSTERING:
    import numpy as np
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from ml.eval.feature_ablation import build_default_feature_sets
    from ml.features.selection import fit_umap_2d
    from ml.eval.reencuadre_plots import (
        plot_umap_clusters, plot_cluster_ndvi_curves,
    )

    feature_sets_for_cluster = build_default_feature_sets(fused.columns)
    pheno_only_cols = feature_sets_for_cluster.get('phenology_only', ())
    if pheno_only_cols:
        X_pheno = fused.select(list(pheno_only_cols)).to_numpy().astype(np.float64)
        # Imputacion por media de columna para tolerar NaN.
        col_means = np.nanmean(np.where(np.isfinite(X_pheno), X_pheno, np.nan), axis=0)
        col_means = np.where(np.isnan(col_means), 0.0, col_means)
        X_clean = np.where(np.isfinite(X_pheno), X_pheno, col_means)
        X_scaled = StandardScaler().fit_transform(X_clean)
        n_clusters = min(N_CLUSTERS, fused['class_id'].n_unique())
        kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
        cluster_labels = kmeans.fit_predict(X_scaled)
        display(Markdown(f'**Clustering**: KMeans con `n_clusters={n_clusters}` sobre `{X_scaled.shape}`.'))
        # UMAP 2D.
        embedding = fit_umap_2d(X_scaled, random_state=RANDOM_STATE)
        np.savez_compressed(
            env.reports_dir / 'umap_embedding.npz',
            embedding=embedding, cluster_labels=cluster_labels,
        )
        fig_umap = plot_umap_clusters(
            embedding, cluster_labels,
            title='UMAP de la firma fenológica pura, coloreado por cluster KMeans',
        )
        fig_umap.savefig(env.figures_dir / 'umap_clusters.png', bbox_inches='tight')
        display(fig_umap)
        plt.close(fig_umap)
        # Curva NDVI media por cluster.
        fig_curves = plot_cluster_ndvi_curves(
            fused, cluster_labels, sequence_length=72,
            title='Curva NDVI media reconstruida por cluster (sin coordenadas)',
        )
        fig_curves.savefig(env.figures_dir / 'cluster_ndvi_curves.png', bbox_inches='tight')
        display(fig_curves)
        plt.close(fig_curves)
        # Composicion top-3 cultivos por cluster.
        df_with_clusters = fused.with_columns(
            pl.Series('pheno_cluster', cluster_labels).cast(pl.Int64)
        )
        top_per_cluster = (
            df_with_clusters.group_by(['pheno_cluster', 'class_id']).len()
            .sort(['pheno_cluster', 'len'], descending=[False, True])
            .group_by('pheno_cluster', maintain_order=True).head(3)
        )
        top_per_cluster.write_parquet(env.reports_dir / 'cluster_class_counts.parquet')
        display(Markdown('**Top-3 cultivos por cluster**:'))
        display(top_per_cluster)
    else:
        display(Markdown('> Sin columnas `phenology_only` detectadas en `fused`; clustering omitido.'))
else:
    display(Markdown('> ENABLE_CLUSTERING=False: bloque omitido.'))


### 6.1 Interpretación agronómica de los clusters

Los clusters de KMeans sobre la firma fenológica pura tienden a corresponder a **arquetipos estacionales**, no a regiones:

- Pico temprano (DOY 80-120) + senescencia temprana → cultivos de invierno (trigo, cebada).
- Pico tardío (DOY 180-220) + maduración larga → cultivos de verano largos (maíz, girasol).
- Pico bajo y área bajo curva pequeña → cultivos de ciclo corto, suelos desnudos parte del año o cultivos minoritarios.

El gráfico de curvas NDVI medias por cluster es el diagnóstico clave: si dos clusters tienen curvas claramente distintas (pico en distinto DOY, amplitud distinta), la fenología los está separando sin ayuda del contexto.

## 7. Estrategia para el desbalance ~31× max/min

El desbalance es la causa principal del techo en F1-macro. Tres opciones consideradas:

1. **Pesos por clase** (`class_weight='balanced'` en RF y `sample_weight` inverso a la frecuencia en XGB/LGBM). Ya activado en el baseline; bajo costo, evidencia mixta.
2. **Oversampling sintético (SMOTE) o duplicación aleatoria**. Riesgo de leakage espacial vía vecinos sintéticos — descartado en este dataset.
3. **Fusión de clases minoritarias en una macro-clase `other_minor`**. Sacrifica granularidad pero suele estabilizar F1-macro. Es la decisión recomendada para la siguiente fase si las clases débiles del diagnóstico de la sección 5 siguen rindiendo F1=0.

In [ ]:
class_counts = (
    fused.group_by('class_id').len().sort('len', descending=True)
    .with_columns((pl.col('len') / fused.height * 100).round(2).alias('pct'))
)
imbalance_ratio = float(class_counts['len'].max()) / max(
    float(class_counts['len'].min()), 1.0
)
weak_classes = (
    class_counts.filter(pl.col('len') < WEAK_CLASS_THRESHOLD)
    .get_column('class_id').to_list()
)
display(Markdown(
    f'**Imbalance ratio max/min**: `{imbalance_ratio:.1f}x`. '
    f'**Clases débiles** (< `{WEAK_CLASS_THRESHOLD}` parcelas): `{weak_classes}` '
    f'({len(weak_classes)} clases).'
))
display(Markdown(
    '**Estrategia recomendada**: mantener pesos por clase (opción 1) '
    'y evaluar fusión en macro-clase `other_minor` (opción 3) si '
    'las clases débiles siguen con F1 ≈ 0 en el diagnóstico de la sección 5.'
))
class_counts.write_parquet(env.reports_dir / 'class_counts.parquet')


## 8. Rama semántica fenológica — descripción textual real con Gemini Flash

El paper Wen et al. (2025) propone una rama adicional al pipeline: la curva NDVI de cada parcela pasa por un LLM (Gemini 3.5 Flash) que produce una descripción estructurada en lenguaje natural — por ejemplo *cultivo de verano con pico medio en julio, senescencia abrupta en septiembre*. Un text-encoder convierte ese texto en un vector denso de 384 dimensiones que se concatena al vector tabular como bloque opcional `pheno_text_*`.

Esta sección **no mockea** el LLM: si `GEMINI_API_KEY` está presente y `ENABLE_LLM_BRANCH=True`, lanza Gemini real sobre un subset balanceado de `LLM_SUBSET_SIZE = 1080` parcelas (60 por clase × 18 clases). El bloque resultante se persiste y compara contra la corrida histórica `with_pheno_text` de la sección 3.

In [ ]:
if ENABLE_LLM_BRANCH and env.has_gemini_api_key:
    from ml.features.phenology_description import build_phenology_text_block

    # Subset balanceado: 60 parcelas por clase.
    per_class = max(1, LLM_SUBSET_SIZE // fused['class_id'].n_unique())
    rng_seed = RANDOM_STATE
    balanced = (
        fused.with_columns(pl.lit(np.random.default_rng(rng_seed).random(fused.height)).alias('__r'))
        .group_by('class_id')
        .map_groups(lambda g: g.sort('__r').head(min(per_class, g.height)))
        .drop('__r')
    )
    pheno_ndvi_cols = ['parcel_id', 'year'] + [c for c in balanced.columns if c.startswith('NDVI_fft')]
    cols_present = [c for c in pheno_ndvi_cols if c in balanced.columns]
    display(Markdown(
        f'**Subset balanceado LLM**: `{balanced.height}` parcelas '
        f'({per_class}/clase × {fused["class_id"].n_unique()} clases).'
    ))
    with track_experiment(
        experiment_name=MLFLOW_EXPERIMENT,
        run_name='05-llm-pheno-text-real',
        tracking_uri=mlflow_uri,
        probe_server=False,
    ) as run:
        import mlflow
        text_block = build_phenology_text_block(
            balanced.select(cols_present),
            skip_llm=False,
            cache_dir=env.repo / 'data/cache/phenology_descriptions',
        )
        mlflow.log_params({
            'llm': 'gemini-3.5-flash',
            'n_parcels': balanced.height,
            'per_class': per_class,
            'embedding_dim': text_block.shape[1] - 1,
        })
        MLFLOW_RUN_IDS['llm-pheno-text'] = run.info.run_id
        display(Markdown(
            f'**Bloque `pheno_text`** real: shape=`{text_block.shape}` '
            f'(run `{run.info.run_id[:12]}...`).'
        ))
elif ENABLE_LLM_BRANCH:
    display(Markdown(
        '> `GEMINI_API_KEY` ausente. Define la variable en `.env.local` '
        'antes de activar `ENABLE_LLM_BRANCH=True`. Saltamos el bloque.'
    ))
else:
    display(Markdown('> ENABLE_LLM_BRANCH=False: rama LLM omitida.'))


## 9. Conclusiones consolidadas — decisión por bloque

**Lo que validamos numéricamente en este cuaderno**:

1. **Las features geográficas (`geom_*`) se descartan** del baseline. La sección 3 confirma la decisión de `04c`: `geom_only` < 0.10 (no clasifican por sí solas) y `no_geom` no degrada respecto a `full`.
2. **Clima y topografía crudos son redundantes** con AlphaEarth. La diferencia entre `no_geom` y `no_geom_no_era5_srtm` es marginal — el embedding fundacional ya los codifica.
3. **La firma fenológica explícita lleva una parte importante de la señal**. `phenology_only` queda cerca del conjunto completo.
4. **Los modelos temporales consumen mejor la información** que el resumen anual — la sección 4 compara los 5 modelos sobre el mismo conjunto ganador y muestra si TempCNN/InceptionTime superan a XGBoost.
5. **La estructura existe sin coordenadas**. KMeans sobre la firma fenológica pura agrupa parcelas por arquetipo estacional. Las curvas NDVI medias por cluster son interpretables agronómicamente.

**Decisiones por bloque opcional** (umbral `delta >= +0.005`):

- **FarSLIP**: si `with_farslip - full >= +0.005`, promover; si delta en [-0.005, +0.005], diferir a stacking (Avance 5); si <-0.005, descartar.
- **pheno_text (Gemini Flash real)**: misma regla. La rama semántica de la sección 8 produce un parquet sobre 1080 parcelas balanceadas.
- **Firma espectral REP (Frampton 2013)**: misma regla.

**Sobre las preguntas oficiales**:

- **P1**: la tabla de la sección 4 cubre **5 modelos reales** (RF, XGB, LGBM, TempCNN, InceptionTime). El ganador final lo decide `Avance3.Equipo17.ipynb`.
- **P2**: ablación con bloques opcionales + decisión por umbral cuantifica el aporte real.
- **P3**: diagnóstico sub/sobreajuste para los 2 temporales leyendo loss desde MLflow (sección 5.1).

## Lo que sigue

`Avance3.Equipo17.ipynb` lee `ablation_table.parquet` + `model_comparison_temporal.parquet` + las tablas de los notebooks 04, 04b, 04c, 04_farslip; ejecuta `select_winning_features()` (única vez en el proyecto) y persiste el conjunto ganador.

## 10. Glosario

- **Ablation**: experimento que entrena el mismo modelo sobre varios subconjuntos de features para medir cuánto aporta cada bloque. Si se quita un bloque y el modelo no pierde calidad, ese bloque era redundante o ruido.
- **Spatial CV**: en lugar de dividir las parcelas al azar entre folds, se asegura que las geográficamente cercanas vayan al mismo fold y se respeta un buffer de separación. Evita que el modelo memorice la ubicación en lugar del cultivo.
- **Out-of-fold (OOF)**: predicción sobre una parcela obtenida en el fold donde esa parcela quedó en validación. Por construcción, el conjunto OOF reúne predicciones honestas sobre el dataset completo.
- **F1-macro**: promedio simple del F1 por clase, sin ponderar por soporte. Penaliza con fuerza fallos en clases minoritarias — la métrica natural cuando importa rendir en todas las clases por igual.
- **mIoU (mean Intersection over Union)**: promedio de Jaccard por clase. Equivalente a F1-macro en sensibilidad al desbalance, más estricto.
- **FFT (Fast Fourier Transform)**: descomposición de la serie temporal NDVI en armónicos. Los primeros capturan la estacionalidad anual; los posteriores, picos cortos. Permite resumir 72 puntos en 8 números (4 amplitudes + 4 fases) por índice espectral.
- **Fenología (phenology)**: estudio de los eventos estacionales del cultivo (emergencia, pico, senescencia, cosecha). Las features fenológicas describen esos eventos como atributos derivados de la curva NDVI.
- **REP (Red Edge Position)**: longitud de onda (nm) del punto de inflexión entre el rojo y el infrarrojo cercano de Sentinel-2 (Frampton 2013). Cambia con el contenido de clorofila y la fenología.
- **TempCNN / InceptionTime**: modelos temporales 1D para series temporales. Importados de `breizhcrops.models`. Ven la curva completa, no su resumen anual.
- **FarSLIP / RemoteCLIP**: extractores de embeddings visuales basados en CLIP, afinados para teledetección. Generan un vector denso por parcela que se usa como bloque opcional del baseline tabular o como modelo independiente.